# Groundswarm — Ollama host (Colab)

Runs Ollama + `dolphin3` on a Colab GPU runtime and exposes it over a public
`cloudflared` quick tunnel, so the local `groundswarm` code (unchanged) can
point at it instead of `localhost:11434`.

**Before running:** Runtime -> Change runtime type -> GPU (T4 is fine for dolphin3).

**Model is pinned to `dolphin3:latest`** — the same model every prior local run
used. Do not swap models for these comparisons: changing environment AND
model at the same time reintroduces the exact confound this move exists to
remove (see ADR-013 / `html/decisions/adr-index.html`).

Run all 4 cells top to bottom, then copy the printed URL into a local shell:

```powershell
$env:OLLAMA_HOST = "https://<whatever-was-printed>.trycloudflare.com"
```

before running any `groundswarm` scenario script. `OllamaClient` reads
`OLLAMA_HOST` at import time (see `src/groundswarm/llm/ollama_client.py`),
so no code changes are needed on the local side.

**Keep this tab open** for the duration of the run — the tunnel and the
Ollama server both die when the Colab runtime disconnects or is recycled.

In [ ]:
# 1. Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# 2. Start the server in the background, wait for it, then pull the model
import subprocess, time, urllib.request, urllib.error

ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

for _ in range(30):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2)
        print("Ollama server is up.")
        break
    except (urllib.error.URLError, OSError):
        time.sleep(1)
else:
    raise RuntimeError("Ollama server did not come up in 30s -- check the cell above for errors.")

!ollama pull dolphin3:latest

In [ ]:
# 3. Sanity check: confirm the model actually answers before exposing it publicly
import json, urllib.request

req = urllib.request.Request(
    "http://127.0.0.1:11434/v1/chat/completions",
    data=json.dumps({
        "model": "dolphin3:latest",
        "messages": [{"role": "user", "content": "Reply with exactly: ok"}],
    }).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST",
)
with urllib.request.urlopen(req, timeout=60) as resp:
    data = json.loads(resp.read().decode("utf-8"))
print(data["choices"][0]["message"]["content"])

In [ ]:
# 4. Install cloudflared and open a quick tunnel (no account/authtoken needed)
import subprocess, re, threading

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

tunnel_proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:11434"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

public_url = None
def _watch():
    global public_url
    for line in tunnel_proc.stdout:
        m = re.search(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com", line)
        if m and public_url is None:
            public_url = m.group(0)
            print(f"\n{'='*60}\nPUBLIC OLLAMA URL: {public_url}\n{'='*60}\n")
            print("On your LOCAL machine (PowerShell), before running any groundswarm script:\n")
            print(f'  $env:OLLAMA_HOST = "{public_url}"\n')

threading.Thread(target=_watch, daemon=True).start()
print("Waiting for tunnel URL (usually 5-15s)...")

### Notes

- This exposes an **unauthenticated** Ollama endpoint on the public internet
  for as long as this notebook runs. The URL is a random subdomain (hard to
  guess) but anyone who obtains it can send it prompts and burn your Colab
  GPU quota. Don't share the URL, and stop the runtime when done
  (Runtime -> Disconnect and delete runtime).
- Free-tier Colab GPU sessions are time- and quota-limited and can be
  reclaimed with little warning; a long scale sweep may need to be resumed
  after a fresh tunnel URL if the runtime recycles mid-run.
- To confirm the tunnel is actually serving Ollama (not just up), from your
  local machine: `curl $env:OLLAMA_HOST/api/tags`